In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import PandasTools
from rdkit.Chem.Descriptors import MolLogP
from sklearn.metrics import confusion_matrix,accuracy_score,f1_score
from rdkit.Chem.AllChem import GetMorganFingerprintAsBitVect
from rdkit.Chem import Descriptors
from rdkit.Chem import PandasTools
from rdkit.DataStructs import ExplicitBitVect

import sys
import multiprocessing
from standardiser import break_bonds, neutralise, rules, unsalt
from standardiser.utils import StandardiseException, sanity_check
def warn(*args, **kwargs):
    pass 
import warnings
warnings.filterwarnings("ignore")
warnings.warn = warn
from rdkit.Chem import AllChem as Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
import sys
from sklearn.metrics import cohen_kappa_score
import csv
from rdkit.Chem import MACCSkeys
from sklearn.model_selection import ShuffleSplit
import _pickle as cPickle
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import StratifiedShuffleSplit    
import bz2
from glob import glob
import _pickle as cPickle
import pickle



In [2]:
def load_sdf_to_df(filename):
    suppl = Chem.SDMolSupplier(filename)
    rows = []
    for mol in suppl:
        if mol is not None:
            row = {prop: mol.GetProp(prop) for prop in mol.GetPropNames()}
            row['SMILES'] = Chem.MolToSmiles(mol)
            rows.append(row)
    return pd.DataFrame(rows)

# Load the  test sets from SDF files
test_df = load_sdf_to_df(r'C:\Users\User\Documents\KULIAH TELKOM S1\SKRIPSI\UJI COBA QSAR - PRIBADI\Cardiac toxicity\dataset\test_set_acute_cardiotoxicity_features.sdf')

# Convert strings back to lists of integers
def string_to_list(bit_string):
    if isinstance(bit_string, str):
        return list(map(int, bit_string.strip('[]').split(', ')))
    else:
        return bit_string


test_df['Morgan_Descriptors'] = test_df['Morgan_Descriptors'].apply(string_to_list)
test_df['MACCS_Descriptors'] = test_df['MACCS_Descriptors'].apply(string_to_list)

def string_to_list(descriptor):
    if isinstance(descriptor, str):
        return list(map(float, descriptor.strip('[]').split(',')))
    return descriptor

# Apply the function to the 'Modred_Descriptor' column
test_df['Modred_Descriptor'] = test_df['Modred_Descriptor'].apply(string_to_list)

# Convert 'Modred_Descriptor' column to a NumPy array
data_modred_test = np.array(test_df['Modred_Descriptor'].tolist())


print("Test DataFrame:")
print(test_df.head())


Test DataFrame:
                                              smiles labels  \
0  Cc1c(Cl)ccc(OC2CCN(C[C@H](O)CNC(=O)c3c[nH]c(=O...      0   
1  Cc1ccc(C(=O)N2CCN(c3ccc(OCCCN4CCCCC4)cc3)C(=O)...      0   
2  O=C(Nc1ccc(F)cn1)[C@H](COCCO)Oc1ncnc2c1cnn2-c1...      0   
3  CC(=O)NC[C@@H]1OC(=O)N2c3cc(F)c(C4=CCN(C(=O)CO...      0   
4  CCC(NCCCC[C@@H](C[C@@H](OC)c1ccc(F)cc1)C(=O)NO...      1   

                                          IUPAC name ID  \
0  N-[(2R)-3-[4-(2,4-dichloro-3-methylphenoxy)pip...      
1  4-(4-methylbenzoyl)-1-[4-(3-piperidin-1-ylprop...      
2  (2S)-2-[1-(2-chlorophenyl)pyrazolo[3,4-d]pyrim...      
3  N-[[(3S,3aS)-8-fluoro-7-[1-(2-hydroxyacetyl)-3...      
4  (2S)-2-[(2R)-2-(4-fluorophenyl)-2-methoxyethyl...      

                                  Morgan_Descriptors  \
0  [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   
1  [0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...   
2  [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...   
3  [0, 0, 0, 0, 0, 0, 0, 0

In [3]:
test_df

,smiles,labels,IUPAC name,ID,Morgan_Descriptors,MACCS_Descriptors,Modred_Descriptor,SMILES
0,Cc1c(Cl)ccc(OC2CCN(C[C@H](O)CNC(=O)c3c[nH]c(=O...,0,"N-[(2R)-3-[4-(2,4-dichloro-3-methylphenoxy)pip...",,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.690806048294513, 0.2924079450017158, 0.5112...",Cc1c(Cl)ccc(OC2CCN(C[C@H](O)CNC(=O)c3c[nH]c(=O...
1,Cc1ccc(C(=O)N2CCN(c3ccc(OCCCN4CCCCC4)cc3)C(=O)...,0,4-(4-methylbenzoyl)-1-[4-(3-piperidin-1-ylprop...,,"[0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.31562202906715986, -0.2696430986656145, -0...",Cc1ccc(C(=O)N2CCN(c3ccc(OCCCN4CCCCC4)cc3)C(=O)...
2,O=C(Nc1ccc(F)cn1)[C@H](COCCO)Oc1ncnc2c1cnn2-c1...,0,"(2S)-2-[1-(2-chlorophenyl)pyrazolo[3,4-d]pyrim...",,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.9467144649001344, 0.9353224603156779, 0.702...",O=C(Nc1ccc(F)cn1)[C@H](COCCO)Oc1ncnc2c1cnn2-c1...
3,CC(=O)NC[C@@H]1OC(=O)N2c3cc(F)c(C4=CCN(C(=O)CO...,0,"N-[[(3S,3aS)-8-fluoro-7-[1-(2-hydroxyacetyl)-3...",,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.6615367703519848, 0.39523433487335896, 0.37...",CC(=O)NC[C@@H]1OC(=O)N2c3cc(F)c(C4=CCN(C(=O)CO...
4,CCC(NCCCC[C@@H](C[C@@H](OC)c1ccc(F)cc1)C(=O)NO...,1,(2S)-2-[(2R)-2-(4-fluorophenyl)-2-methoxyethyl...,,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.027280088254953223, -0.503248683422551, -0...",CCC(NCCCC[C@@H](C[C@@H](OC)c1ccc(F)cc1)C(=O)NO...
...,...,...,...,...,...,...,...,...
2063,Cc1ccc(-c2c(C)c(CNC3CCCC3F)nn2-c2ncccc2Cl)cn1,0,N-[[1-(3-chloropyridin-2-yl)-4-methyl-5-(6-met...,,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.5430522292962664, -0.34950825584747336, -0...",Cc1ccc(-c2c(C)c(CNC3CCCC3F)nn2-c2ncccc2Cl)cn1
2064,COC[C@@H](NC(C)=O)C(=O)NCc1ccccc1,0,(2R)-2-acetamido-N-benzyl-3-methoxypropanamide,,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-1.4689897923159863, -1.9228518523300888, -1....",COC[C@@H](NC(C)=O)C(=O)NCc1ccccc1
2065,CC(=O)NC(c1ccccc1-c1cc(Cl)cc(Cl)c1)C(c1cccnc1)...,0,"N-[1-[2-(3,5-dichlorophenyl)phenyl]-2,2-dipyri...",,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.40204550515422044, -0.14185884717464053, -...",CC(=O)NC(c1ccccc1-c1cc(Cl)cc(Cl)c1)C(c1cccnc1)...
2066,CC(C)(NCc1cc([C@]2(C)CCSC(N)=N2)c(F)cc1F)C(F)(F)F,0,"(4S)-4-[2,4-difluoro-5-[[(1,1,1-trifluoro-2-me...",,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.16850764933358223, -0.5212183437884692, -0....",CC(C)(NCc1cc([C@]2(C)CCSC(N)=N2)c(F)cc1F)C(F)(F)F


In [4]:
test_df = test_df.rename(columns={'labels': 'Outcome'})

In [5]:
test_df= test_df.sort_values(['Outcome'], ascending=True)
test_df['RowID'] = test_df.index
test_df.head(100)

,smiles,Outcome,IUPAC name,ID,Morgan_Descriptors,MACCS_Descriptors,Modred_Descriptor,SMILES,RowID
0,Cc1c(Cl)ccc(OC2CCN(C[C@H](O)CNC(=O)c3c[nH]c(=O...,0,"N-[(2R)-3-[4-(2,4-dichloro-3-methylphenoxy)pip...",,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.690806048294513, 0.2924079450017158, 0.5112...",Cc1c(Cl)ccc(OC2CCN(C[C@H](O)CNC(=O)c3c[nH]c(=O...,0
800,CCOc1c([C@H](C)n2nc(C)c3c(N)ncnc32)cc(Cl)c(F)c...,0,"(4R)-4-[3-[(1S)-1-(4-amino-3-methylpyrazolo[3,...",,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.17780262273425018, 0.39623264933813196, 0.3...",CCOc1c([C@H](C)n2nc(C)c3c(N)ncnc32)cc(Cl)c(F)c...,800
801,Cc1cc(Cl)ccc1OC1CCN(C[C@H](O)CNC(=O)c2c[nH]c(=...,0,N-[(2R)-3-[4-(4-chloro-2-methylphenoxy)piperid...,,"[0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.4341065701228783, 0.35130849842333645, 0.15...",Cc1cc(Cl)ccc1OC1CCN(C[C@H](O)CNC(=O)c2c[nH]c(=...,801
802,Cc1c([C@H]2CN3CCN(C(=O)Cc4ccc(-n5cnnn5)cn4)C[C...,0,"3-[(3S,9aS)-8-[2-[5-(tetrazol-1-yl)pyridin-2-y...",,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.8858027243170344, 0.9972179571316186, 0.539...",Cc1c([C@H]2CN3CCN(C(=O)Cc4ccc(-n5cnnn5)cn4)C[C...,802
1634,CC1(C)Cc2c(CN3CCC4(CC3)CCN(C(=O)c3ccc(N)cn3)CC...,0,"(5-aminopyridin-2-yl)-[9-[(2,2-dimethyl-3H-1-b...",,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.4437740027614739, -0.16182513647010524, -0...",CC1(C)Cc2c(CN3CCC4(CC3)CCN(C(=O)c3ccc(N)cn3)CC...,1634
...,...,...,...,...,...,...,...,...,...
927,N#Cc1ccc2cc1Oc1cccc(c1)CN1CC[C@H](NCc3cncn3C2)...,0,"(6S)-27-oxo-20-oxa-3,7,11,13-tetrazapentacyclo...",,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.075337078390321, 0.16162875011642233, -0.0...",N#Cc1ccc2cc1Oc1cccc(c1)CN1CC[C@H](NCc3cncn3C2)...,927
1587,O=C(Nc1ccccc1)Nc1n[nH]c2nnc(-c3ccccc3)c(-c3ccc...,0,"1-(4,5-diphenyl-1H-pyrazolo[3,4-c]pyridazin-3-...",,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.020776901880414544, 0.5659461083495818, 0.3...",O=C(Nc1ccccc1)Nc1n[nH]c2nnc(-c3ccccc3)c(-c3ccc...,1587
891,COc1ccc2c(c1)N(CCN1CCC(NCc3ccc4c(n3)NC(=O)CO4)...,0,"6-[[[1-[2-(7-methoxy-2-oxo-3,4-dihydroquinolin...",,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.3091188426926212, 0.41320399523927714, 0.12...",COc1ccc2c(c1)N(CCN1CCC(NCc3ccc4c(n3)NC(=O)CO4)...,891
890,O=C(CCNC(=O)c1cccc([N+](=O)[O-])c1)NC1CCOC1=O,0,3-nitro-N-[3-oxo-3-[(2-oxooxolan-3-yl)amino]pr...,,"[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.139413065237478, -0.7009149474476513, -0.7...",O=C(CCNC(=O)c1cccc([N+](=O)[O-])c1)NC1CCOC1=O,890


In [6]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()


outcomes=(np.unique(test_df['Outcome']))
le.fit(list(set(outcomes)))
y = le.transform( test_df['Outcome'] )



print ("Classes                          : ",(outcomes))
print ("Number of cpds in each class     : ",np.unique([len(y[y==smi]) for smi in y]))
print ("Total number of cpds             : ",len(y))

S = test_df['Outcome']
info = {}
for i,cls in enumerate(S.unique()):
    info.update({cls:i})
    S = S.replace(cls,i)

Classes                          :  ['0' '1']
Number of cpds in each class     :  [ 904 1164]
Total number of cpds             :  2068


In [7]:
print(test_df['Outcome'].value_counts())

1    1164
0     904
Name: Outcome, dtype: int64


In [8]:
#load model - all included
import joblib

# Base paths
fingerprint_path = r"C:\Users\User\Documents\KULIAH TELKOM S1\SKRIPSI\UJI COBA QSAR - PRIBADI\Cardiac toxicity\model\fingerprint descriptor model"
phys_path        = r"C:\Users\User\Documents\KULIAH TELKOM S1\SKRIPSI\UJI COBA QSAR - PRIBADI\Cardiac toxicity\model\physical chemical properties model"

# descriptor
rf_morgan   = joblib.load(fr"{fingerprint_path}\Model_Cardiac_RF_morgan.pkl")  
rf_maccs    = joblib.load(fr"{fingerprint_path}\Model_Cardiac_RF_maccs.pkl")
rf_modred   = joblib.load(fr"{fingerprint_path}\Model_Cardiac_RF_modred.pkl")

svm_morgan  = joblib.load(fr"{fingerprint_path}\Model_Cardiac_SVM_morgan.pkl")  
svm_maccs   = joblib.load(fr"{fingerprint_path}\Model_Cardiac_SVM_maccs.pkl")
svm_modred  = joblib.load(fr"{fingerprint_path}\Model_Cardiac_SVM_modred.pkl")

xgb_morgan  = joblib.load(fr"{fingerprint_path}\Model_Cardiac_XGB_morgan.pkl")  
xgb_maccs   = joblib.load(fr"{fingerprint_path}\Model_Cardiac_XGB_maccs.pkl")
xgb_modred  = joblib.load(fr"{fingerprint_path}\Model_Cardiac_XGB_modred.pkl")

nn_morgan   = joblib.load(fr"{fingerprint_path}\Model_Cardiac_NN_morgan.pkl")  
nn_maccs    = joblib.load(fr"{fingerprint_path}\Model_Cardiac_NN_maccs.pkl")
nn_modred   = joblib.load(fr"{fingerprint_path}\Model_Cardiac_NN_modred.pkl")

lgbm_morgan = joblib.load(fr"{fingerprint_path}\Model_Cardiac_LGBM_morgan.pkl")  
lgbm_maccs  = joblib.load(fr"{fingerprint_path}\Model_Cardiac_LGBM_maccs.pkl")
lgbm_modred = joblib.load(fr"{fingerprint_path}\Model_Cardiac_LGBM_modred.pkl")

# physiochemical
rf_phys   = joblib.load(fr"{phys_path}\Model_Cardiac_RF_phys.pkl")
svm_phys  = joblib.load(fr"{phys_path}\Model_Cardiac_SVM_phys.pkl")
xgb_phys  = joblib.load(fr"{phys_path}\Model_Cardiac_XGB_phys.pkl")
nn_phys   = joblib.load(fr"{phys_path}\Model_Cardiac_NN_phys.pkl")
lgbm_phys = joblib.load(fr"{phys_path}\Model_Cardiac_LGBM_phys.pkl")


In [9]:
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, f1_score, classification_report


In [10]:
y_true = test_df['Outcome'].astype(int)  # Ensure it's of integer type, suitable for metrics calculation


In [11]:
# RF
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    f1_score
)

# Prediction Function 
def predict_probabilities_rf_from_pkl(test_df, rf_morgan, rf_maccs, rf_modred):

    X_morgan = pd.DataFrame(
        test_df['Morgan_Descriptors'].tolist()
    ).iloc[:, :rf_morgan.n_features_in_]

    X_maccs = pd.DataFrame(
        test_df['MACCS_Descriptors'].tolist()
    ).iloc[:, :rf_maccs.n_features_in_]

    X_modred = pd.DataFrame(
        test_df['Modred_Descriptor'].tolist()
    ).iloc[:, :rf_modred.n_features_in_]

    return {
        "Morgan": rf_morgan.predict_proba(X_morgan)[:, 1],
        "MACCS":  rf_maccs.predict_proba(X_maccs)[:, 1],
        "Modred": rf_modred.predict_proba(X_modred)[:, 1]
    }

# Evaluation Function
def evaluate_model(y_true, y_prob, threshold=0.5):

    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    return {
        "Confusion Matrix": cm,
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1 Score": f1_score(y_true, y_pred),
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "CCR": (sensitivity + specificity) / 2,
        "PPV (Precision)": tp / (tp + fp) if (tp + fp) else 0.0,
        "NPV": tn / (tn + fn) if (tn + fn) else 0.0
    }


# Consensus Function 
def consensus_probability(probs_dict):
    probs = np.vstack(list(probs_dict.values()))
    return probs.mean(axis=0)

probs_rf = predict_probabilities_rf_from_pkl(
    test_df,
    rf_morgan,
    rf_maccs,
    rf_modred
)

# Results
for model_name, y_prob in probs_rf.items():
    print(f"\n{model_name} RF Performance")
    metrics = evaluate_model(y_true, y_prob)
    for k, v in metrics.items():
        print(f"{k}: {v}")

consensus_probs_rf = consensus_probability(probs_rf)
consensus_metrics_rf = evaluate_model(y_true, consensus_probs_rf)

print("\nCONSENSUS RF (Morgan + MACCS + Modred)")
for k, v in consensus_metrics_rf.items():
    print(f"{k}: {v}")



Morgan RF Performance
Confusion Matrix: [[607 297]
 [168 996]]
Accuracy: 0.7751450676982592
AUC: 0.8690461256576346
F1 Score: 0.8107448107448108
Sensitivity: 0.8556701030927835
Specificity: 0.6714601769911505
CCR: 0.763565140041967
PPV (Precision): 0.7703016241299304
NPV: 0.7832258064516129

MACCS RF Performance
Confusion Matrix: [[636 268]
 [184 980]]
Accuracy: 0.781431334622824
AUC: 0.8525026229358634
F1 Score: 0.8126036484245439
Sensitivity: 0.8419243986254296
Specificity: 0.7035398230088495
CCR: 0.7727321108171396
PPV (Precision): 0.7852564102564102
NPV: 0.775609756097561

Modred RF Performance
Confusion Matrix: [[ 631  273]
 [ 155 1009]]
Accuracy: 0.793036750483559
AUC: 0.8727752562114162
F1 Score: 0.8250204415372036
Sensitivity: 0.8668384879725086
Specificity: 0.6980088495575221
CCR: 0.7824236687650153
PPV (Precision): 0.7870514820592823
NPV: 0.8027989821882952

CONSENSUS RF (Morgan + MACCS + Modred)
Confusion Matrix: [[ 640  264]
 [ 161 1003]]
Accuracy: 0.7944874274661509
AUC: 

In [12]:
# SVM
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    f1_score
)

def predict_probabilities_from_pkl(test_df, svm_morgan, svm_maccs, svm_modred):

    X_morgan = pd.DataFrame(
        test_df['Morgan_Descriptors'].tolist()
    ).iloc[:, :svm_morgan.n_features_in_]

    X_maccs = pd.DataFrame(
        test_df['MACCS_Descriptors'].tolist()
    ).iloc[:, :svm_maccs.n_features_in_]

    X_modred = pd.DataFrame(
        test_df['Modred_Descriptor'].tolist()
    ).iloc[:, :svm_modred.n_features_in_]

    return {
        "Morgan": svm_morgan.predict_proba(X_morgan)[:, 1],
        "MACCS":  svm_maccs.predict_proba(X_maccs)[:, 1],
        "Modred": svm_modred.predict_proba(X_modred)[:, 1]
    }

def evaluate_model(y_true, y_prob, threshold=0.5):

    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    return {
        "Confusion Matrix": cm,
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1 Score": f1_score(y_true, y_pred),
        "Sensitivity (Recall+)": sensitivity,
        "Specificity (Recall-)": specificity,
        "CCR": (sensitivity + specificity) / 2,
        "PPV (Precision)": tp / (tp + fp) if (tp + fp) else 0.0,
        "NPV": tn / (tn + fn) if (tn + fn) else 0.0
    }

def consensus_probability(probs_dict):
    probs = np.vstack(list(probs_dict.values()))
    return probs.mean(axis=0)

probs_svm = predict_probabilities_from_pkl(
    test_df,
    svm_morgan,
    svm_maccs,
    svm_modred
)

for model_name, y_prob in probs_svm.items():
    print(f"\n{model_name} SVM Performance")
    metrics = evaluate_model(y_true, y_prob)
    for k, v in metrics.items():
        print(f"{k}: {v}")

consensus_probs_svm = consensus_probability(probs_svm)
consensus_metrics_svm = evaluate_model(y_true, consensus_probs_svm)

print("\nCONSENSUS SVM (Morgan + MACCS + Modred)")
for k, v in consensus_metrics_svm.items():
    print(f"{k}: {v}")



Morgan SVM Performance
Confusion Matrix: [[643 261]
 [169 995]]
Accuracy: 0.7920696324951644
AUC: 0.8717911800930571
F1 Score: 0.8223140495867769
Sensitivity (Recall+): 0.8548109965635738
Specificity (Recall-): 0.7112831858407079
CCR: 0.7830470912021409
PPV (Precision): 0.7921974522292994
NPV: 0.791871921182266

MACCS SVM Performance
Confusion Matrix: [[648 256]
 [195 969]]
Accuracy: 0.7819148936170213
AUC: 0.8500459964115195
F1 Score: 0.8112180828798661
Sensitivity (Recall+): 0.8324742268041238
Specificity (Recall-): 0.7168141592920354
CCR: 0.7746441930480796
PPV (Precision): 0.7910204081632654
NPV: 0.7686832740213523

Modred SVM Performance
Confusion Matrix: [[638 266]
 [182 982]]
Accuracy: 0.7833655705996132
AUC: 0.8659684525438676
F1 Score: 0.814262023217247
Sensitivity (Recall+): 0.8436426116838488
Specificity (Recall-): 0.7057522123893806
CCR: 0.7746974120366147
PPV (Precision): 0.7868589743589743
NPV: 0.7780487804878049

CONSENSUS SVM (Morgan + MACCS + Modred)
Confusion Matrix:

In [13]:
# XGB
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    f1_score
)

# Prediction Function 
def predict_probabilities_from_pkl(test_df, xgb_morgan, xgb_maccs, xgb_modred):

    X_morgan = pd.DataFrame(
        test_df['Morgan_Descriptors'].tolist()
    ).iloc[:, :xgb_morgan.n_features_in_]

    X_maccs = pd.DataFrame(
        test_df['MACCS_Descriptors'].tolist()
    ).iloc[:, :xgb_maccs.n_features_in_]

    X_modred = pd.DataFrame(
        test_df['Modred_Descriptor'].tolist()
    ).iloc[:, :xgb_modred.n_features_in_]

    return {
        "Morgan": xgb_morgan.predict_proba(X_morgan)[:, 1],
        "MACCS":  xgb_maccs.predict_proba(X_maccs)[:, 1],
        "Modred": xgb_modred.predict_proba(X_modred)[:, 1]
    }

# Evaluation Function
def evaluate_model(y_true, y_prob, threshold=0.5):

    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    return {
        "Confusion Matrix": cm,
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1 Score": f1_score(y_true, y_pred),
        "Sensitivity (Recall+)": sensitivity,
        "Specificity (Recall-)": specificity,
        "CCR": (sensitivity + specificity) / 2,
        "PPV (Precision)": tp / (tp + fp) if (tp + fp) else 0.0,
        "NPV": tn / (tn + fn) if (tn + fn) else 0.0
    }

# Consensus Function 
def consensus_probability(probs_dict):
    probs = np.vstack(list(probs_dict.values()))
    return probs.mean(axis=0)

probs_xgb = predict_probabilities_from_pkl(
    test_df,
    xgb_morgan,
    xgb_maccs,
    xgb_modred
)

# Results
for model_name, y_prob in probs_xgb.items():
    print(f"\n{model_name} XGB Performance")
    metrics = evaluate_model(y_true, y_prob)
    for k, v in metrics.items():
        print(f"{k}: {v}")
consensus_probs_xgb = consensus_probability(probs_xgb)
consensus_metrics_xgb = evaluate_model(y_true, consensus_probs_xgb)

print("\nCONSENSUS XGB (Morgan + MACCS + Modred)")
for k, v in consensus_metrics_xgb.items():
    print(f"{k}: {v}")




Morgan XGB Performance
Confusion Matrix: [[648 256]
 [171 993]]
Accuracy: 0.7935203094777563
AUC: 0.870875528388529
F1 Score: 0.823041856610029
Sensitivity (Recall+): 0.8530927835051546
Specificity (Recall-): 0.7168141592920354
CCR: 0.7849534713985951
PPV (Precision): 0.7950360288230585
NPV: 0.7912087912087912

MACCS XGB Performance
Confusion Matrix: [[657 247]
 [214 950]]
Accuracy: 0.7770793036750484
AUC: 0.8516682252227594
F1 Score: 0.8047437526471835
Sensitivity (Recall+): 0.8161512027491409
Specificity (Recall-): 0.7267699115044248
CCR: 0.7714605571267829
PPV (Precision): 0.7936507936507936
NPV: 0.7543053960964409

Modred XGB Performance
Confusion Matrix: [[670 234]
 [196 968]]
Accuracy: 0.7920696324951644
AUC: 0.8767153620411763
F1 Score: 0.8182586644125106
Sensitivity (Recall+): 0.8316151202749141
Specificity (Recall-): 0.7411504424778761
CCR: 0.7863827813763951
PPV (Precision): 0.8053244592346089
NPV: 0.7736720554272517

CONSENSUS XGB (Morgan + MACCS + Modred)
Confusion Matrix:

In [14]:
# NN 
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    f1_score
)

# Prediction Function 
def predict_probabilities_from_pkl(test_df, nn_morgan, nn_maccs, nn_modred):

    X_morgan = pd.DataFrame(
        test_df['Morgan_Descriptors'].tolist()
    ).iloc[:, :nn_morgan.n_features_in_]

    X_maccs = pd.DataFrame(
        test_df['MACCS_Descriptors'].tolist()
    ).iloc[:, :nn_maccs.n_features_in_]

    X_modred = pd.DataFrame(
        test_df['Modred_Descriptor'].tolist()
    ).iloc[:, :nn_modred.n_features_in_]

    return {
        "Morgan": nn_morgan.predict_proba(X_morgan)[:, 1],
        "MACCS":  nn_maccs.predict_proba(X_maccs)[:, 1],
        "Modred": nn_modred.predict_proba(X_modred)[:, 1]
    }

# Evaluation Function
def evaluate_model(y_true, y_prob, threshold=0.5):

    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    return {
        "Confusion Matrix": cm,
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1 Score": f1_score(y_true, y_pred),
        "Sensitivity (Recall+)": sensitivity,
        "Specificity (Recall-)": specificity,
        "CCR": (sensitivity + specificity) / 2,
        "PPV (Precision)": tp / (tp + fp) if (tp + fp) else 0.0,
        "NPV": tn / (tn + fn) if (tn + fn) else 0.0
    }

# Consensus Function 
def consensus_probability(probs_dict):
    probs = np.vstack(list(probs_dict.values()))
    return probs.mean(axis=0)

probs = predict_probabilities_from_pkl(
    test_df,
    nn_morgan,
    nn_maccs,
    nn_modred
)

# Results
for model_name, y_prob in probs.items():
    print(f"\n{model_name} Performance")
    metrics = evaluate_model(y_true, y_prob)
    for k, v in metrics.items():
        print(f"{k}: {v}")

consensus_probs_nn = consensus_probability(probs)
consensus_metrics_nn = evaluate_model(y_true, consensus_probs_nn)

print("\n CONSENSUS NN (Morgan + MACCS + Modred) ")
for k, v in consensus_metrics_nn.items():
    print(f"{k}: {v}")



Morgan Performance
Confusion Matrix: [[647 257]
 [203 961]]
Accuracy: 0.7775628626692457
AUC: 0.8504204300094274
F1 Score: 0.8068849706129303
Sensitivity (Recall+): 0.8256013745704467
Specificity (Recall-): 0.7157079646017699
CCR: 0.7706546695861083
PPV (Precision): 0.7889983579638752
NPV: 0.7611764705882353

MACCS Performance
Confusion Matrix: [[653 251]
 [242 922]]
Accuracy: 0.761605415860735
AUC: 0.8391859965635738
F1 Score: 0.789045785194694
Sensitivity (Recall+): 0.7920962199312714
Specificity (Recall-): 0.7223451327433629
CCR: 0.7572206763373172
PPV (Precision): 0.7860187553282183
NPV: 0.729608938547486

Modred Performance
Confusion Matrix: [[656 248]
 [194 970]]
Accuracy: 0.7862669245647969
AUC: 0.8506456603716206
F1 Score: 0.8144416456759026
Sensitivity (Recall+): 0.8333333333333334
Specificity (Recall-): 0.7256637168141593
CCR: 0.7794985250737463
PPV (Precision): 0.7963875205254516
NPV: 0.7717647058823529

 CONSENSUS NN (Morgan + MACCS + Modred) 
Confusion Matrix: [[674 230]


In [15]:
# LGBM - Consensus
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    f1_score
)

# ==============================
# Prediction Function (PKL-based)
# ==============================
def predict_probabilities_from_pkl(test_df, lgbm_morgan, lgbm_maccs, lgbm_modred):

    X_morgan = pd.DataFrame(
        test_df['Morgan_Descriptors'].tolist()
    ).iloc[:, :lgbm_morgan.n_features_in_]

    X_maccs = pd.DataFrame(
        test_df['MACCS_Descriptors'].tolist()
    ).iloc[:, :lgbm_maccs.n_features_in_]

    X_modred = pd.DataFrame(
        test_df['Modred_Descriptor'].tolist()
    ).iloc[:, :lgbm_modred.n_features_in_]

    return {
        "Morgan": lgbm_morgan.predict_proba(X_morgan)[:, 1],
        "MACCS":  lgbm_maccs.predict_proba(X_maccs)[:, 1],
        "Modred": lgbm_modred.predict_proba(X_modred)[:, 1]
    }

# ==============================
# Evaluation Function
# ==============================
def evaluate_model(y_true, y_prob, threshold=0.5):

    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    return {
        "Confusion Matrix": cm,
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1 Score": f1_score(y_true, y_pred),
        "Sensitivity (Recall+)": sensitivity,
        "Specificity (Recall-)": specificity,
        "CCR": (sensitivity + specificity) / 2,
        "PPV (Precision)": tp / (tp + fp) if (tp + fp) else 0.0,
        "NPV": tn / (tn + fn) if (tn + fn) else 0.0
    }

# ==============================
# Consensus Function (Neutral)
# ==============================
def consensus_probability(probs_dict):
    probs = np.vstack(list(probs_dict.values()))
    return probs.mean(axis=0)

# ==============================
# RUN PREDICTION
# ==============================
probs = predict_probabilities_from_pkl(
    test_df,
    lgbm_morgan,
    lgbm_maccs,
    lgbm_modred
)

# ==============================
# PRINT INDIVIDUAL MODEL RESULTS
# ==============================
for model_name, y_prob in probs.items():
    print(f"\n===== {model_name} Performance =====")
    metrics = evaluate_model(y_true, y_prob)
    for k, v in metrics.items():
        print(f"{k}: {v}")

# ==============================
# CONSENSUS RESULT
# ==============================
consensus_probs_lgbm = consensus_probability(probs)
consensus_metrics_lgbm = evaluate_model(y_true, consensus_probs_lgbm)

print("\n===== CONSENSUS LGBM (Morgan + MACCS + Modred) =====")
for k, v in consensus_metrics_lgbm.items():
    print(f"{k}: {v}")



===== Morgan Performance =====
Confusion Matrix: [[679 225]
 [213 951]]
Accuracy: 0.7882011605415861
AUC: 0.8693131709393911
F1 Score: 0.8128205128205128
Sensitivity (Recall+): 0.8170103092783505
Specificity (Recall-): 0.7511061946902655
CCR: 0.784058251984308
PPV (Precision): 0.8086734693877551
NPV: 0.7612107623318386

===== MACCS Performance =====
Confusion Matrix: [[701 203]
 [237 927]]
Accuracy: 0.7872340425531915
AUC: 0.8623890003953409
F1 Score: 0.8081952920662598
Sensitivity (Recall+): 0.7963917525773195
Specificity (Recall-): 0.7754424778761062
CCR: 0.7859171152267128
PPV (Precision): 0.820353982300885
NPV: 0.7473347547974414

===== Modred Performance =====
Confusion Matrix: [[700 204]
 [186 978]]
Accuracy: 0.811411992263056
AUC: 0.891907482589788
F1 Score: 0.8337595907928389
Sensitivity (Recall+): 0.8402061855670103
Specificity (Recall-): 0.7743362831858407
CCR: 0.8072712343764255
PPV (Precision): 0.8274111675126904
NPV: 0.7900677200902935

===== CONSENSUS LGBM (Morgan + MACC

In [16]:
# ==============================
# CONSENSUS ALL (RF + SVM + XGB + NN + LGBM)
# ==============================
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    f1_score
)

# ==============================
# Evaluation Function
# ==============================
def evaluate_model(y_true, y_prob, threshold=0.5):

    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    return {
        "Confusion Matrix": cm,
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1 Score": f1_score(y_true, y_pred),
        "Sensitivity (Recall+)": sensitivity,
        "Specificity (Recall-)": specificity,
        "CCR": (sensitivity + specificity) / 2,
        "PPV (Precision)": tp / (tp + fp) if (tp + fp) else 0.0,
        "NPV": tn / (tn + fn) if (tn + fn) else 0.0
    }

# ==============================
# STACK ALL MODEL CONSENSUS
# ==============================
all_model_probs = np.vstack([
    consensus_probs_rf,
    consensus_probs_svm,
    consensus_probs_xgb,
    consensus_probs_nn,
    consensus_probs_lgbm
])

# ==============================
# FINAL CONSENSUS (MEAN)
# ==============================
consensus_all_probs = all_model_probs.mean(axis=0)

# ==============================
# EVALUATION
# ==============================
consensus_all_metrics = evaluate_model(y_true, consensus_all_probs)

print("CONSENSUS DESKRIPTOR (RF + SVM + XGB + NN + LGBM)")
for k, v in consensus_all_metrics.items():
    print(f"{k}: {v}")


CONSENSUS DESKRIPTOR (RF + SVM + XGB + NN + LGBM)
Confusion Matrix: [[680 224]
 [166 998]]
Accuracy: 0.811411992263056
AUC: 0.898301363926649
F1 Score: 0.8365465213746857
Sensitivity (Recall+): 0.8573883161512027
Specificity (Recall-): 0.7522123893805309
CCR: 0.8048003527658668
PPV (Precision): 0.8166939443535188
NPV: 0.8037825059101655
